# Text Analytics Coursework

This notebook provides some example code for loading and examining the dataset for task 3. 

In [1]:
pip install fsspec==2023.9.2

Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

# Task 3 - arXiv abstracts

This dataset contains arXiv abstracts over a many year. Let's first load it and examine the fields:

In [3]:
from datasets import load_dataset

# Load the arXiv abstracts dataset
dataset = load_dataset("gfissore/arxiv-abstracts-2021", split="train")

print(dataset)

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'abstract', 'report-no', 'categories', 'versions'],
    num_rows: 1999486
})


We can now filter it to get abstracts about a particular topic or field:

In [4]:
# -----------------------
# Optional: filter by category
# -----------------------
field = "cs.LG"  # Example: Machine Learning
def filter_category(example):
    return field in example['categories']

filtered = dataset.filter(filter_category)

# -----------------------
# Remove missing/short abstracts
# -----------------------
def valid_abstract(example):
    return example['abstract'] is not None and len(example['abstract']) > 50

filtered = filtered.filter(valid_abstract)

Filter:   0%|          | 0/1999486 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7016 [00:00<?, ? examples/s]

In [10]:
print(f"Number of abstracts: {len(docs)}")
print(f"Year range: {min(years)} - {max(years)}")
print(f"\nExample abstract:\n{docs[0]}")

Number of abstracts: 7015
Year range: 1999 - 2021

Example abstract:
  This paper uncovers and explores the close relationship between Monte Carlo
Optimization of a parametrized integral (MCO), Parametric machine-Learning
(PL), and `blackbox' or `oracle'-based optimization (BO). We make four
contributions. First, we prove that MCO is mathematically identical to a broad
class of PL problems. This identity potentially provides a new application
domain for all broadly applicable PL techniques: MCO. Second, we introduce
immediate sampling, a new version of the Probability Collectives (PC) algorithm
for blackbox optimization. Immediate sampling transforms the original BO
problem into an MCO problem. Accordingly, by combining these first two
contributions, we can apply all PL techniques to BO. In our third contribution
we validate this way of improving BO by demonstrating that cross-validation and
bagging improve immediate sampling. Finally, conventional MC and MCO procedures
ignore the rela

Another useful step is to get the date of each abstract, e.g., for plotting trends. This is not stored in the dataset object, but we can infer it from the ID. 

In [8]:
# -----------------------
# Infer year from arXiv ID
# -----------------------
def extract_year(arxiv_id):
    if arxiv_id.startswith("cs/"):
        arxiv_id = arxiv_id.split("/")[1]
    yy = int(arxiv_id[:2])
    return 1900 + yy if yy >= 91 else 2000 + yy

years = [extract_year(x) for x in filtered['id']]

# -----------------------
# Prepare docs
# -----------------------
docs = filtered['abstract']

In [9]:
df = pd.DataFrame({
    'id': filtered['id'],
    'title': filtered['title'],
    'abstract': filtered['abstract'],
    'categories': filtered['categories'],
    'year': years  # your computed list
})

df.to_csv('arxiv_csLG_filtered.csv', index=False)
print(df.head())
print(f"\nShape: {df.shape}")

          id                                              title  \
0  0704.1274   Parametric Learning and Monte Carlo Optimization   
1  0704.2668  Supervised Feature Selection via Dependence Es...   
2  0705.1585  HMM Speaker Identification Using Linear and No...   
3  0706.3679  Scale-sensitive Psi-dimensions: the Capacity M...   
4  0707.3390  Consistency of the group Lasso and multiple ke...   

                                            abstract categories  year  
0    This paper uncovers and explores the close r...    [cs.LG]  2007  
1    We introduce a framework for filtering featu...    [cs.LG]  2007  
2    Speaker identification is a powerful, non-in...    [cs.LG]  2007  
3    Bounds on the risk play a crucial role in st...    [cs.LG]  2007  
4    We consider the least-square regression prob...    [cs.LG]  2007  

Shape: (7015, 5)
